# Preparation soil carbon data

In [ ]:
# Libraries
import xarray as xr
import matplotlib.pyplot as plt

In [ ]:
# Directories
dir01 = '../paper_ecosystem_carbon/output/01_data_prep/'
dir02 = '../paper_ecosystem_carbon/output/02_database_prep/'

---

In [ ]:
# Get data
ds_soilgrids2017 = xr.open_zarr(f'{dir01}ds_prep_soilgrids2017_r001.zarr')
ds_soilgrids2020 = xr.open_zarr(f'{dir01}ds_prep_soilgrids2020_r001.zarr')
ds_gsoc = xr.open_zarr(f'{dir01}ds_prep_gsoc_ocs_r001.zarr')
ds_wise = xr.open_zarr(f'{dir01}ds_prep_wise_r001.zarr')

In [ ]:
# Create array with 0-30cm soil carbon data
da_csoil_000_030 = xr.merge([ds_soilgrids2020['soilgrids2020_ocs_0-30cm'],
                             ds_gsoc['gsoc_ocs']]) \
    .to_array('a').mean('a')

# Create array with 30-100cm soil carbon data
da_csoil_030_100 = xr.merge([ds_soilgrids2017['soilgrids2017_ocs_30-100cm'],
                             ds_wise['wise_soc_30-100cm']]) \
    .to_array('a').mean('a')

# Create array with 0-100cm soil carbon data
da_csoil_000_100 = (da_csoil_000_030 + da_csoil_030_100).rename('csoil_000_100')

# Export array as netcdf for regridding
# Attributes of latitude and longitude
da_csoil_000_100.lat.attrs = dict(standard_name = 'latitude',
                                  long_name = 'latitude',
                                  units = 'degrees_north',
                                  axis = 'Y')
da_csoil_000_100.lon.attrs = dict(standard_name = 'longitude',
                                  long_name = 'longitude',
                                  units = 'degrees_east',
                                  axis = 'X')

# Export
da_csoil_000_100.to_dataset() \
    .to_zarr(f'{dir02}ds_csoil_000_100_r001.zarr', mode='w');

In [ ]:
# Plot to check
ds = xr.open_mfdataset(f'{dir02}ds_csoil_000_100_r001.zarr', engine='zarr')

for i in range(0, len(ds.data_vars)):
    v = list(ds.data_vars)[i]
    fig, ax = plt.subplots(figsize=(10, 5), ncols=1, nrows=1)
    ds[v].plot.imshow(ax=ax, robust=True)
    ax.set_title(v)